# Profiling (Scalene)

Run CPU/GPU bitset scripts and save `scalene_report_*.json` files for plotting.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPEATS = 3
MAKE_HTML = True
CLEAR_OLD = False

scalene_bin = os.environ.get('SCALENE_BIN')
if scalene_bin:
    SCALENE_CMD = [scalene_bin]
elif shutil.which('scalene'):
    SCALENE_CMD = ['scalene']
else:
    SCALENE_CMD = [sys.executable, '-m', 'scalene']

test_files = [
    ('cpu_bitset', 'pandas_cpu_bitset.py', 'pandas_cpu_bitset'),
    ('gpu_bitset', 'pandas_gpu_bitset.py', 'pandas_gpu_bitset'),
]

base_dir = Path.cwd()
if not (base_dir / 'pandas_cpu_bitset.py').exists():
    base_dir = (Path.cwd() / 'notebooks' / 'profiling').resolve()

repo_src = (base_dir / '..' / '..' / 'src').resolve()
os.environ['PYTHONPATH'] = str(repo_src) + os.pathsep + os.environ.get('PYTHONPATH', '')

print('base_dir:', base_dir)
print('PYTHONPATH prefix:', repo_src)
print('scalene command:', ' '.join(SCALENE_CMD))


In [ ]:
for mode, script_name, output_folder in test_files:
    print(f'\n{mode}')
    script_path = base_dir / script_name
    report_dir = base_dir / output_folder
    report_dir.mkdir(parents=True, exist_ok=True)

    if CLEAR_OLD:
        for old_report in report_dir.glob('scalene_report_*.*'):
            old_report.unlink()

    for i in range(REPEATS):
        print(i)
        json_path = report_dir / f'scalene_report_{i}.json'
        html_path = report_dir / f'scalene_report_{i}.html'

        subprocess.run(
            SCALENE_CMD + ['run', f'--outfile={json_path}', str(script_path)],
            check=True,
            cwd=str(base_dir),
        )

        if MAKE_HTML:
            subprocess.run(
                SCALENE_CMD + ['view', '--standalone', f'--outfile={html_path}', str(json_path)],
                check=True,
                cwd=str(base_dir),
            )
